# Prédiction des recettes communales françaises par méthodes de statistical learning

Ce notebook implémente le pipeline complet d'analyse :
- Chargement et exploration des données
- Construction des lags N-1
- Preprocessing (variance threshold, imputation, corrélation, standardisation)
- Entraînement de 6 modèles sur 5 variables cibles
- Analyse SHAP
- Fine-tuning par validation croisée temporelle
- Analyse sans lags (ablation)

In [2]:
pip install shap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 78.4 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 23.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [shap]3/4 [shap]]te]
Note: you may need to restart the kernel to use updated packages.


In [3]:
# ------------------------------------------------------------
# Bloc 0 — Imports et configuration


import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import time
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, loguniform

plt.rcParams.update({
    'font.family'      : 'serif',
    'font.size'        : 11,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.alpha'       : 0.3,
    'grid.linestyle'   : '--',
})

BLUE  = '#2C5F8A'
GREEN = '#2A8A5A'

print('Imports OK')

Imports OK


/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
# ------------------------------------------------------------
# Bloc 1 — Fonctions utilitaires
# ------------------------------------------------------------
# On définit ici les deux fonctions réutilisées dans tout
# le notebook : evaluate() pour les métriques et
# build_pipeline() pour le preprocessing.
# build_pipeline() est volontairement générique : elle prend
# n'importe quelle target_col et retourne X_tr, X_te, y_tr,
# y_te, feat_fin — ce qui permet de l'appeler pour chacune
# des 5 cibles sans dupliquer le code.
# ------------------------------------------------------------

def evaluate(name, y_true, y_pred):
    r2   = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    print(f'{name:30s} | R2={r2:.4f} | RMSE={rmse:.4f} | MAE={mae:.4f}')
    return {'Modele': name, 'R2': round(r2,4),
            'RMSE': round(rmse,4), 'MAE': round(mae,4)}


def build_pipeline(train_df, test_df, target_col, features,
                   forced_features, var_threshold=0.01,
                   corr_threshold=0.95):
    """
    Pipeline complet de preprocessing.

    Etapes :
    1. Filtre NaN sur la cible (on ne peut pas imputer y)
    2. Variance threshold sur features (hors forced_features)
    3. Imputation médiane — fit sur train uniquement
    4. Correlation filter — fit sur train uniquement
    5. Standardisation z-score — fit sur train uniquement

    forced_features : ces variables ne peuvent pas être
    supprimées par les filtres automatiques (lags des cibles).
    """
    tr = train_df[train_df[target_col].notna()].reset_index(drop=True)
    te = test_df[test_df[target_col].notna()].reset_index(drop=True)
    print(f'  Train : {len(tr):,} | Test : {len(te):,}')

    X_tr_raw = tr[features].copy()
    X_te_raw = te[features].copy()

    to_filter = [c for c in features if c not in forced_features]
    forced    = [c for c in features if c in forced_features]

    vt = VarianceThreshold(threshold=var_threshold)
    vt.fit(X_tr_raw[to_filter])
    feat_var = [f for f, k in zip(to_filter, vt.get_support()) if k]
    feat_var = list(dict.fromkeys(feat_var + forced))
    print(f'  Apres variance threshold : {len(feat_var)}')

    imp = SimpleImputer(strategy='median')
    imp.fit(X_tr_raw[feat_var])
    X_tr_imp = pd.DataFrame(
        imp.transform(X_tr_raw[feat_var]), columns=feat_var)
    X_te_imp = pd.DataFrame(
        imp.transform(X_te_raw[feat_var]), columns=feat_var)

    corr_m = X_tr_imp.corr().abs()
    upper  = corr_m.where(
        np.triu(np.ones(corr_m.shape), k=1).astype(bool))
    drop   = [c for c in upper.columns
              if any(upper[c] > corr_threshold)
              and c not in forced_features]
    feat_fin = list(dict.fromkeys(
        [c for c in feat_var if c not in drop]))
    print(f'  Apres correlation filter : {len(feat_fin)}')

    sc = StandardScaler()
    sc.fit(X_tr_imp[feat_fin])
    X_tr = pd.DataFrame(
        sc.transform(X_tr_imp[feat_fin]), columns=feat_fin)
    X_te = pd.DataFrame(
        sc.transform(X_te_imp[feat_fin]), columns=feat_fin)

    y_tr = tr[target_col]
    y_te = te[target_col]

    print(f'  NaN X_train : {X_tr.isnull().sum().sum()} | '
          f'NaN y_train : {y_tr.isnull().sum()}')
    print(f'  Lags forces : '
          f'{[c for c in forced_features if c in feat_fin]}')

    return X_tr, X_te, y_tr, y_te, feat_fin


def run_models(X_tr, X_te, y_tr, y_te, feat, label):
    """
    Entraine 6 modeles et retourne les resultats + les objets.
    Chaque modele est instancie dans la boucle pour eviter
    le bug de partage d'etat entre les appels successifs.
    """
    print(f'\n{"="*60}')
    print(f'MODELES -- {label}')
    print(f'{"="*60}')

    results = []
    models  = {}

    model_defs = [
        ('OLS',
         LinearRegression()),
        ('Ridge',
         Ridge(alpha=1.0)),
        ('Lasso',
         Lasso(alpha=0.001, max_iter=5000)),
        ('Random Forest',
         RandomForestRegressor(n_estimators=200, n_jobs=-1,
                               random_state=42, max_depth=20)),
        ('Extra Trees',
         ExtraTreesRegressor(n_estimators=200, n_jobs=-1,
                             random_state=42, max_depth=20)),
        ('Gradient Boosting',
         GradientBoostingRegressor(n_estimators=200,
                                   learning_rate=0.1,
                                   max_depth=5,
                                   random_state=42,
                                   subsample=0.8)),
    ]

    for name, model in model_defs:
        print(f'Training {name}...')
        t0 = time.time()
        model.fit(X_tr, y_tr)
        results.append(evaluate(name, y_te, model.predict(X_te)))
        models[name] = model
        print(f'   temps : {time.time()-t0:.1f}s')

    res_df = pd.DataFrame(results).sort_values('R2', ascending=False)
    print(f'\nTABLEAU {label}')
    print(res_df.to_string(index=False))

    print(f'\nOVERFITTING CHECK {label}')
    for name, m in models.items():
        r2_tr = r2_score(y_tr, m.predict(X_tr))
        r2_te = r2_score(y_te, m.predict(X_te))
        flag  = 'WARNING' if r2_tr - r2_te > 0.05 else 'OK'
        print(f'{flag} {name:25s} | '
              f'train={r2_tr:.4f} | test={r2_te:.4f} | '
              f'gap={r2_tr-r2_te:.4f}')

    print(f'\nTOP 15 FEATURES -- Extra Trees {label}')
    imp_s = pd.Series(
        models['Extra Trees'].feature_importances_,
        index=feat
    ).sort_values(ascending=False)
    print(imp_s.head(15))

    return res_df, models


print('Fonctions definies')

Fonctions definies


In [6]:
# ------------------------------------------------------------
# Bloc 2 — Chargement et exploration initiale
# ------------------------------------------------------------
# On charge le parquet et on vérifie les 3 cibles principales.
# Les 35 401 NaN sur les variables ofgl correspondent à 2016 :
# les comptes OFGL ne sont disponibles qu'à partir de 2017,
# donc toute l'année 2016 sera exclue à l'étape des lags.
# ------------------------------------------------------------

# Sur Colab, décommenter :
# from google.colab import files
# uploaded = files.upload()

df = pd.read_parquet('/home/onyxia/ML/data/base_clean.parquet', engine='pyarrow')

print('=== DIMENSIONS ===')
print(f'Lignes : {df.shape[0]:,} | Colonnes : {df.shape[1]}')
print(f'Annees : {sorted(df["an"].unique())}')
print(f'Communes uniques : {df["codgeo"].nunique():,}')

targets = [
    'ofgl_Recettes_totales',
    'ofgl_Recettes_de_fonctionnement',
    'ofgl_Recettes_d_investissement_hors_emprunts'
]

print('\n=== VERIFICATION DES CIBLES ===')
for t in targets:
    n_miss = df[t].isnull().sum()
    print(f'  {t}')
    print(f'    NaN : {n_miss:,} ({n_miss/len(df)*100:.1f}%) | '
          f'mediane : {df[t].median():,.0f} euros')

print('\n=== FAMILLES DE VARIABLES ===')
for prefix, label in [('ofgl_', 'Comptes OFGL'),
                       ('dot_',  'Dotations'),
                       ('geo_',  'Geographie'),
                       ('dvf_',  'Immobilier DVF')]:
    cols = [c for c in df.columns if c.startswith(prefix)]
    pct  = df[cols].isnull().mean().mean() * 100 if cols else 0
    print(f'  {label:20s} : {len(cols):3d} vars | {pct:.1f}% NaN')

=== DIMENSIONS ===
Lignes : 279,920 | Colonnes : 364
Annees : [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Communes uniques : 34,990

=== VERIFICATION DES CIBLES ===
  ofgl_Recettes_totales
    NaN : 35,401 (12.6%) | mediane : 476,092 euros
  ofgl_Recettes_de_fonctionnement
    NaN : 35,401 (12.6%) | mediane : 362,849 euros
  ofgl_Recettes_d_investissement_hors_emprunts
    NaN : 35,401 (12.6%) | mediane : 60,880 euros

=== FAMILLES DE VARIABLES ===
  Comptes OFGL         :  52 vars | 13.2% NaN
  Dotations            : 261 vars | 0.0% NaN
  Geographie           :  16 vars | 0.0% NaN
  Immobilier DVF       :   8 vars | 12.5% NaN


In [7]:
# ------------------------------------------------------------
# Bloc 3 — Création des lags N-1
# ------------------------------------------------------------
# On crée les lags avec pd.concat pour éviter le
# PerformanceWarning lié aux insertions répétées de colonnes.
# Le tri par ['codgeo', 'an'] est indispensable avant groupby
# pour que shift(1) corresponde bien à l'année précédente.
# On exclut ensuite 2016 car ses lags sont tous NaN
# (pas de données 2015 dans le panel).
# ------------------------------------------------------------

ids = ['codgeo', 'an', 'code_dept',
       'geo_reg_code', 'geo_dep_code', 'geo_epci_code',
       'geo_code_postal', 'geo_zone_emploi',
       'aav2020_libgeo', 'geo_typecom']

ofgl_to_lag = [c for c in df.columns
               if c.startswith('ofgl_')
               and c not in targets
               and not c.endswith('_log')]

dot_to_lag = [c for c in df.columns
              if c.startswith('dot_')
              and df[c].nunique() > 10]

exog_cols = [c for c in df.columns
             if (c.startswith('geo_') or
                 c.startswith('dvf_') or
                 c in ['part_alf','nb_als','part_apl','zonage_afr22',
                       'aav2020','ti_zpp2','dens_pop','ind_vieillist',
                       'etctot','p_log','nais_an','nivcentr',
                       'part_men_voit1p','part_men_anem0209',
                       'part_men_anem2m','part_men_anem_10p',
                       'part_rp','part_resid2','pvd_zpp','p_pop',
                       'alloc_rsa','tx_var_pop','va_zpp'])
             and c not in ids]

print(f'Variables ofgl a lagger     : {len(ofgl_to_lag)}')
print(f'Variables dot a lagger      : {len(dot_to_lag)}')
print(f'Variables exogenes (no lag) : {len(exog_cols)}')

df_sorted  = df.sort_values(['codgeo', 'an'])
lag_frames = {}
for col in ofgl_to_lag + dot_to_lag + targets:
    lag_frames[col + '_lag1'] = (
        df_sorted.groupby('codgeo')[col].shift(1)
    )

df_model = pd.concat(
    [df_sorted, pd.DataFrame(lag_frames)], axis=1
)
lag_cols = list(lag_frames.keys())

# Transformation log de toutes les cibles
all_targets = [
    'ofgl_Recettes_totales',
    'ofgl_Recettes_de_fonctionnement',
    'ofgl_Recettes_d_investissement_hors_emprunts',
    'ofgl_Recettes_totales_hors_emprunts',
]
for t in all_targets:
    if t in df_model.columns:
        df_model[t + '_log'] = np.log1p(df_model[t])

# Cible B : recettes par habitant
df_model['ofgl_Recettes_totales_pct_hab'] = np.log1p(
    df_model['ofgl_Recettes_totales'] /
    df_model['geo_population'].replace(0, np.nan)
)

# Exclure 2016 + defragmentation
df_model = df_model[df_model['an'] >= 2017].copy()

print(f'\nLags crees : {len(lag_cols)}')
print(f'Shape df_model : {df_model.shape}')
print('Etape 3 terminee')

Variables ofgl a lagger     : 49
Variables dot a lagger      : 182
Variables exogenes (no lag) : 41

Lags crees : 234
Shape df_model : (244930, 603)
Etape 3 terminee


In [8]:
# ------------------------------------------------------------
# Bloc 4 — Encodage + features candidates + split temporel
# ------------------------------------------------------------
# Le label encoding est préféré au one-hot encoding pour les
# variables catégorielles car les arbres de décision gèrent
# nativement les entiers et ça évite d'exploser la dimension.
#
# Les lags des 3 cibles principales sont protégés (forced) :
# ils ne peuvent pas être éliminés par les filtres automatiques
# même si leur variance est faible après standardisation.
#
# Split temporel strict pour éviter le data leakage :
# train = 2017-2021, test = 2022-2023.
# ------------------------------------------------------------

# Encodage categoriel
cat_to_encode = [c for c in exog_cols
                 if c in df_model.columns
                 and df_model[c].dtype == 'object']
for col in cat_to_encode:
    le = LabelEncoder()
    df_model[col] = df_model[col].fillna('MISSING')
    df_model[col] = le.fit_transform(df_model[col].astype(str))
print(f'Colonnes encodees : {cat_to_encode}')

# Lags des cibles a forcer
lags_cibles_forced = [t + '_lag1' for t in targets]

# Features candidates
all_feat = list(set(
    exog_cols +
    [c for c in lag_cols if c in df_model.columns] +
    lags_cibles_forced
))

feat_ok = []
for c in all_feat:
    if c not in df_model.columns:
        continue
    if df_model[c].dtype not in [np.float64, np.int64, float, int]:
        continue
    if df_model[c].isnull().mean() < 0.30 or c in lags_cibles_forced:
        feat_ok.append(c)
feat_ok = list(dict.fromkeys(feat_ok))
print(f'Features candidates : {len(feat_ok)}')

# Split temporel
train_c = df_model[
    df_model['an'] <= 2021
].copy().reset_index(drop=True)
test_c  = df_model[
    df_model['an'] >= 2022
].copy().reset_index(drop=True)

print(f'Train : {len(train_c):,} | Test : {len(test_c):,}')

Colonnes encodees : ['geo_reg_nom', 'geo_dep_nom', 'geo_epci_nom', 'zonage_afr22', 'ti_zpp2', 'pvd_zpp', 'va_zpp']
Features candidates : 273
Train : 174,950 | Test : 69,980


In [11]:
# ------------------------------------------------------------
# Bloc 5 — Pipelines cible A et cible B
# ------------------------------------------------------------
# Cible A : log(recettes totales) — cible principale
# Cible B : log(recettes/habitant) — cible normalisée
#
# Les deux pipelines sont identiques sauf pour la target_col.
# On garde geo_population dans les features pour la cible B
# (contrairement à une approche naïve qui l'exclurait) car
# la taille peut légitimement modérer les recettes/habitant
# via les économies d'échelle dans les services publics.
# ------------------------------------------------------------

print('=== PIPELINE CIBLE A ===')
X_tr_A, X_te_A, y_tr_A, y_te_A, feat_A = build_pipeline(
    train_c, test_c,
    'ofgl_Recettes_totales_log',
    feat_ok, lags_cibles_forced
)

print('\n=== PIPELINE CIBLE B ===')
X_tr_B, X_te_B, y_tr_B, y_te_B, feat_B = build_pipeline(
    train_c, test_c,
    'ofgl_Recettes_totales_pct_hab',
    feat_ok, lags_cibles_forced
)

=== PIPELINE CIBLE A ===
  Train : 174,655 | Test : 69,864
  Apres variance threshold : 226
  Apres correlation filter : 158
  NaN X_train : 0 | NaN y_train : 0
  Lags forces : ['ofgl_Recettes_totales_lag1', 'ofgl_Recettes_de_fonctionnement_lag1', 'ofgl_Recettes_d_investissement_hors_emprunts_lag1']

=== PIPELINE CIBLE B ===
  Train : 174,625 | Test : 69,852
  Apres variance threshold : 226
  Apres correlation filter : 158
  NaN X_train : 0 | NaN y_train : 0
  Lags forces : ['ofgl_Recettes_totales_lag1', 'ofgl_Recettes_de_fonctionnement_lag1', 'ofgl_Recettes_d_investissement_hors_emprunts_lag1']


In [10]:
# ------------------------------------------------------------
# Bloc 6 — Modèles sur cible A
# ------------------------------------------------------------
# Les réseaux de neurones ont été écartés : sur la cible A
# ils donnaient R2=0.54, moins bien que les modèles linéaires
# (R2=0.74), pour un temps de calcul 20x supérieur.
# Ce résultat est cohérent avec la littérature sur données
# tabulaires où les méthodes à base d'arbres dominent.
# ------------------------------------------------------------

res_A_df, models_A = run_models(
    X_tr_A, X_te_A, y_tr_A, y_te_A,
    feat_A, 'CIBLE A log(recettes totales)'
)

NameError: name 'X_tr_A' is not defined

In [ ]:
# ------------------------------------------------------------
# Bloc 7 — Modèles sur cible B
# ------------------------------------------------------------
# Sur la cible B, GBM domine (R2=0.719) avec un gap
# d'overfitting quasi-nul, alors que RF et Extra Trees
# overfittent significativement (gap ~0.16).
# La régularisation implicite du GBM (learning_rate=0.1
# + subsampling=0.8) est plus adaptée à la structure
# plus diffuse des recettes par habitant.
# ------------------------------------------------------------

res_B_df, models_B = run_models(
    X_tr_B, X_te_B, y_tr_B, y_te_B,
    feat_B, 'CIBLE B log(recettes/habitant)'
)

In [ ]:
# ------------------------------------------------------------
# Bloc 8 — Baseline naïf + vérification sanity check
# ------------------------------------------------------------
# Le baseline naïf prédit recettes(t) = recettes(t-1).
# Son R2 très négatif (-0.39) s'explique par l'hétérogénéité
# de taille : dans l'espace log, une commune passant de
# 200k à 400k euros génère un résidu de log(2)~0.69 par
# rapport au baseline, ce qui gonfle l'erreur quadratique.
# Cela montre que le gain des Extra Trees (+1.36 R2) reflète
# un véritable apprentissage et pas juste de la persistance.
# ------------------------------------------------------------

print('=== BASELINE NAIF ===')
y_naive = np.log1p(test_c['ofgl_Recettes_totales_lag1'])
mask    = y_naive.notna() & y_te_A.reset_index(drop=True).notna()
r2_naive  = r2_score(
    y_te_A.reset_index(drop=True)[mask], y_naive[mask])
rmse_naive = np.sqrt(mean_squared_error(
    y_te_A.reset_index(drop=True)[mask], y_naive[mask]))

print(f'Baseline naif (N=N-1) | R2={r2_naive:.4f} | RMSE={rmse_naive:.4f}')
print(f'Extra Trees           | R2={res_A_df.iloc[0]["R2"]:.4f} | '
      f'RMSE={res_A_df.iloc[0]["RMSE"]:.4f}')
print(f'Gain vs Baseline      | +{res_A_df.iloc[0]["R2"]-r2_naive:.4f} R2')

# Sanity check : vérifier que les prédictions sont dans le bon range
print('\n=== SANITY CHECK CIBLE A ===')
y_pred_A = models_A['Extra Trees'].predict(X_te_A)
print(f'y_te_A range  : {y_te_A.min():.2f} -- {y_te_A.max():.2f}')
print(f'y_pred range  : {y_pred_A.min():.2f} -- {y_pred_A.max():.2f}')
residus_A = y_pred_A - y_te_A.values
print(f'Residus mean={residus_A.mean():.4f} | std={residus_A.std():.4f}')

In [ ]:
# ------------------------------------------------------------
# Bloc 9 — Cibles supplémentaires C, D, E
# ------------------------------------------------------------
# Cible C : recettes totales hors emprunts (plus propre
#           économiquement que A car exclut les emprunts
#           qui sont non récurrents)
# Cible D : recettes de fonctionnement
# Cible E : recettes d'investissement hors emprunts
#           (structurellement plus volatile — R2 attendu < 0.7)
# ------------------------------------------------------------

extra_targets = {
    'Cible C -- Recettes totales hors emprunts':
        'ofgl_Recettes_totales_hors_emprunts',
    'Cible D -- Recettes de fonctionnement':
        'ofgl_Recettes_de_fonctionnement',
    'Cible E -- Recettes investissement hors emprunts':
        'ofgl_Recettes_d_investissement_hors_emprunts'
}

# Transformation log
for label, col in extra_targets.items():
    if col in df_model.columns:
        df_model[col + '_log'] = np.log1p(df_model[col])
        if col in train_c.columns:
            train_c[col + '_log'] = np.log1p(train_c[col])
            test_c[col + '_log']  = np.log1p(test_c[col])

results_extra = {}
models_extra  = {}

for label, col in extra_targets.items():
    target_log = col + '_log'
    print(f'\n{"="*60}')
    print(f'PIPELINE -- {label}')
    print(f'{"="*60}')

    X_tr, X_te, y_tr, y_te, feat = build_pipeline(
        train_c, test_c, target_log, feat_ok, lags_cibles_forced
    )
    res, mods = run_models(X_tr, X_te, y_tr, y_te, feat, label)

    results_extra[label] = res
    models_extra[label]  = mods

    # Sanity check
    y_pred = mods['Extra Trees'].predict(X_te)
    print(f'\nSANITY CHECK')
    print(f'  y_te  : {y_te.min():.2f} -- {y_te.max():.2f}')
    print(f'  pred  : {y_pred.min():.2f} -- {y_pred.max():.2f}')
    res_tmp = y_pred - y_te.values
    print(f'  residus mean={res_tmp.mean():.4f} | std={res_tmp.std():.4f}')

# Tableau récap
print('\n' + '='*70)
print('TABLEAU RECAP -- MEILLEUR MODELE PAR CIBLE')
print('='*70)
recap = []
recap.append({'Cible': 'A -- Recettes totales',
              'Meilleur': res_A_df.iloc[0]['Modele'],
              'R2': res_A_df.iloc[0]['R2'],
              'RMSE': res_A_df.iloc[0]['RMSE']})
recap.append({'Cible': 'B -- Recettes/habitant',
              'Meilleur': res_B_df.iloc[0]['Modele'],
              'R2': res_B_df.iloc[0]['R2'],
              'RMSE': res_B_df.iloc[0]['RMSE']})
for label, res in results_extra.items():
    short = label.split('--')[1].strip()
    recap.append({'Cible': short,
                  'Meilleur': res.iloc[0]['Modele'],
                  'R2': res.iloc[0]['R2'],
                  'RMSE': res.iloc[0]['RMSE']})
print(pd.DataFrame(recap).to_string(index=False))

In [ ]:
# ------------------------------------------------------------
# Bloc 10 — Figures pour le rapport
# ------------------------------------------------------------
# Les figures sont sauvegardées en PNG dans le répertoire
# courant. 
# ------------------------------------------------------------

ordre  = ['OLS','Ridge','Lasso',
          'Gradient Boosting','Random Forest','Extra Trees']
labels = ['OLS','Ridge','Lasso',
          'Gradient\nBoosting','Random\nForest','Extra\nTrees']

r2_A_vals = [res_A_df[res_A_df['Modele']==m]['R2'].values[0]
             for m in ordre]
r2_B_vals = [res_B_df[res_B_df['Modele']==m]['R2'].values[0]
             for m in ordre]

x     = np.arange(len(ordre))
width = 0.35

fig, ax = plt.subplots(figsize=(13, 6))
bars_A = ax.bar(x - width/2, r2_A_vals, width,
                label='Cible A -- log(recettes totales)',
                color=BLUE, edgecolor='white', linewidth=0.5)
bars_B = ax.bar(x + width/2, r2_B_vals, width,
                label='Cible B -- log(recettes/habitant)',
                color=GREEN, edgecolor='white',
                linewidth=0.5, alpha=0.85)

for bar in list(bars_A) + list(bars_B):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.008,
            f'{h:.3f}', ha='center', va='bottom', fontsize=8.5)

ax.axvline(x=2.5, color='grey', linestyle='--',
           linewidth=1, alpha=0.6)
ax.text(1.0, 1.05, 'Modeles lineaires', ha='center',
        fontsize=9, color='grey', style='italic')
ax.text(4.5, 1.05, 'Algorithmes ML', ha='center',
        fontsize=9, color='grey', style='italic')
ax.axhline(y=r2_naive, color='red', linestyle=':',
           linewidth=1.5,
           label=f'Baseline naif (R2={r2_naive:.3f})')

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('$R^2$ hors-echantillon', fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_title(
    'Figure 4 -- Comparaison des performances predictives\n'
    '(ensemble de test 2022-2023)',
    fontsize=12, fontweight='bold', pad=12)
ax.legend(fontsize=9, framealpha=0.5)
plt.tight_layout()
plt.savefig('fig4_comparaison_modeles.png',
            dpi=150, bbox_inches='tight')
plt.close()
print('fig4 sauvegardee')

# Figure 5 — Predicted vs Actual
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_te_A.values, y_pred_A,
                alpha=0.06, color=BLUE, s=2, rasterized=True)
lims = [float(y_te_A.min()), float(y_te_A.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5,
             label='Prediction parfaite')
axes[0].set_xlabel('Valeurs reelles', fontsize=11)
axes[0].set_ylabel('Valeurs predites', fontsize=11)
axes[0].set_title('Predictions vs Realite\n(Extra Trees, Cible A)',
                  fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9)

axes[1].hist(residus_A, bins=80, color=BLUE,
             edgecolor='white', linewidth=0.3)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Residus', fontsize=11)
axes[1].set_ylabel('Frequence', fontsize=11)
axes[1].set_title('Distribution des residus\n(Extra Trees, Cible A)',
                  fontsize=11, fontweight='bold')
axes[1].text(
    0.97, 0.95,
    f'mean={residus_A.mean():.4f}\nstd={residus_A.std():.4f}',
    transform=axes[1].transAxes, ha='right', va='top',
    fontsize=9,
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle(
    'Figure 5 -- Diagnostic des predictions (Extra Trees, Cible A)',
    fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig5_predicted_vs_actual.png',
            dpi=150, bbox_inches='tight')
plt.close()
print('fig5 sauvegardee')

In [ ]:
# ------------------------------------------------------------
# Bloc 11 — SHAP
# ------------------------------------------------------------
# On utilise TreeExplainer qui calcule les valeurs SHAP
# exactement en temps polynomial (pas approché).
# On se limite à 500 observations du test pour réduire
# le temps de calcul (~2-5 min par modèle).
# On utilise GBM pour les deux cibles car TreeExplainer
# est plus rapide sur GBM que sur Extra Trees.
# ------------------------------------------------------------

np.random.seed(42)

# SHAP cible A
print('Calcul SHAP GBM cible A...')
idx_A    = np.random.choice(len(X_te_A), size=500, replace=False)
X_shap_A = X_te_A.iloc[idx_A]
exp_A       = shap.TreeExplainer(models_A['Gradient Boosting'])
shap_vals_A = exp_A.shap_values(X_shap_A)
print('SHAP cible A calcule')

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_vals_A, X_shap_A,
                  max_display=20, show=False)
plt.title(
    'Figure 6 -- SHAP values -- Gradient Boosting\n'
    'Cible A',
    fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('fig6_shap_cibleA.png', dpi=150, bbox_inches='tight')
plt.close()
print('fig6 sauvegardee')

# SHAP cible B
print('\nCalcul SHAP GBM cible B...')
idx_B    = np.random.choice(len(X_te_B), size=500, replace=False)
X_shap_B = X_te_B.iloc[idx_B]
exp_B       = shap.TreeExplainer(models_B['Gradient Boosting'])
shap_vals_B = exp_B.shap_values(X_shap_B)
print('SHAP cible B calcule')

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_vals_B, X_shap_B,
                  max_display=20, show=False)
plt.title(
    'Figure 7 -- SHAP values -- Gradient Boosting\n'
    'Cible B',
    fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('fig7_shap_cibleB.png', dpi=150, bbox_inches='tight')
plt.close()
print('fig7 sauvegardee')

In [ ]:
# ------------------------------------------------------------
# Bloc 12 — Fine tuning (à lancer la nuit)
# ------------------------------------------------------------
# On utilise des folds temporels walk-forward plutôt que
# le KFold classique qui mélangerait les années et créerait
# du data leakage.
# Fold k : train = toutes les années < année_k
#          val   = année_k
# 20 configurations tirées aléatoirement par modèle.
# Temps estimé : 1-2h selon la machine.
# ------------------------------------------------------------

def make_temporal_splits(years, min_train_years=2):
    unique_years = sorted(np.unique(years))
    splits = []
    for i in range(min_train_years, len(unique_years)):
        train_idx = np.where(years < unique_years[i])[0]
        val_idx   = np.where(years == unique_years[i])[0]
        splits.append((train_idx, val_idx))
    return splits

years_A  = train_c[
    train_c['ofgl_Recettes_totales_log'].notna()
]['an'].values
splits_A = make_temporal_splits(years_A)

years_B  = train_c[
    train_c['ofgl_Recettes_totales_pct_hab'].notna()
]['an'].values
splits_B = make_temporal_splits(years_B)

print(f'Folds temporels : {len(splits_A)}')
for i, (tr, val) in enumerate(splits_A):
    print(f'  Fold {i+1} : '
          f'train={sorted(np.unique(years_A[tr]))} '
          f'| val={sorted(np.unique(years_A[val]))}')

# Tuning Extra Trees — cible A
print('\n=== TUNING Extra Trees (Cible A) ===')
param_et = {
    'n_estimators'    : [100, 200, 300, 500],
    'max_depth'       : [10, 15, 20, 30, None],
    'min_samples_leaf': [1, 2, 4, 8],
}
et_search = RandomizedSearchCV(
    ExtraTreesRegressor(n_jobs=-1, random_state=42),
    param_distributions=param_et,
    n_iter=20, cv=splits_A, scoring='r2',
    random_state=42, verbose=1, refit=True
)
t0 = time.time()
et_search.fit(X_tr_A.values, y_tr_A.values)
r2_et_tuned = r2_score(
    y_te_A, et_search.best_estimator_.predict(X_te_A))
print(f'Meilleurs params : {et_search.best_params_}')
print(f'R2 CV            : {et_search.best_score_:.4f}')
print(f'R2 test default  : {res_A_df.iloc[0]["R2"]:.4f}')
print(f'R2 test tuned    : {r2_et_tuned:.4f}')
print(f'Gain             : +{r2_et_tuned-res_A_df.iloc[0]["R2"]:.4f}')
print(f'Temps            : {time.time()-t0:.1f}s')

# Tuning GBM — cible B
print('\n=== TUNING GBM (Cible B) ===')
param_gbm = {
    'n_estimators' : [100, 200, 300],
    'learning_rate': loguniform(0.01, 0.3),
    'max_depth'    : [3, 4, 5, 6, 8],
    'subsample'    : uniform(0.6, 0.4),
}
gbm_search = RandomizedSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_distributions=param_gbm,
    n_iter=20, cv=splits_B, scoring='r2',
    random_state=42, verbose=1, refit=True
)
t0 = time.time()
gbm_search.fit(X_tr_B.values, y_tr_B.values)
r2_gbm_tuned = r2_score(
    y_te_B, gbm_search.best_estimator_.predict(X_te_B))
print(f'Meilleurs params : {gbm_search.best_params_}')
print(f'R2 CV            : {gbm_search.best_score_:.4f}')
print(f'R2 test default  : {res_B_df.iloc[0]["R2"]:.4f}')
print(f'R2 test tuned    : {r2_gbm_tuned:.4f}')
print(f'Gain             : +{r2_gbm_tuned-res_B_df.iloc[0]["R2"]:.4f}')
print(f'Temps            : {time.time()-t0:.1f}s')

In [ ]:
# ------------------------------------------------------------
# Bloc 13 — Ablation : modèles sans lags
# ------------------------------------------------------------
# On refait tourner les modèles en utilisant uniquement
# les variables exogènes (geo, dvf, observatoire) sans
# aucun lag de variables comptables.
# L'objectif est de quantifier la valeur ajoutée des lags :
# combien de R2 perd-on si on ne dispose que de données
# structurelles sur la commune ?
# ------------------------------------------------------------

print('=== ABLATION : MODELES SANS LAGS ===')
print('Features : uniquement variables exogenes')
print('(geo, dvf, observatoire — pas de lags ofgl/dot/cibles)\n')

# Pipeline sans lags : on passe forced_features=[] pour ne
# rien forcer et on utilise uniquement exog_cols
feat_exog_num = [c for c in exog_cols
                 if c in df_model.columns
                 and df_model[c].dtype in
                 [np.float64, np.int64, float, int]]

print('=== PIPELINE CIBLE A SANS LAGS ===')
X_tr_A_nl, X_te_A_nl, y_tr_A_nl, y_te_A_nl, feat_A_nl = \
    build_pipeline(
        train_c, test_c,
        'ofgl_Recettes_totales_log',
        feat_exog_num, []  # forced_features vide
    )

print('\n=== PIPELINE CIBLE B SANS LAGS ===')
X_tr_B_nl, X_te_B_nl, y_tr_B_nl, y_te_B_nl, feat_B_nl = \
    build_pipeline(
        train_c, test_c,
        'ofgl_Recettes_totales_pct_hab',
        feat_exog_num, []
    )

res_A_nl, _ = run_models(
    X_tr_A_nl, X_te_A_nl, y_tr_A_nl, y_te_A_nl,
    feat_A_nl, 'CIBLE A sans lags'
)
res_B_nl, _ = run_models(
    X_tr_B_nl, X_te_B_nl, y_tr_B_nl, y_te_B_nl,
    feat_B_nl, 'CIBLE B sans lags'
)

# Tableau comparatif
print('\n' + '='*70)
print('COMPARAISON AVEC / SANS LAGS')
print('='*70)
for label, res_avec, res_sans in [
    ('Cible A', res_A_df, res_A_nl),
    ('Cible B', res_B_df, res_B_nl)
]:
    print(f'\n{label}')
    print(f'{"Modele":25s} | {"R2 avec lags":>12} | '
          f'{"R2 sans lags":>12} | {"Gain lags":>10}')
    print('-'*65)
    for m in ['OLS','Ridge','Lasso',
              'Random Forest','Extra Trees','Gradient Boosting']:
        r2_a = float(res_avec[
            res_avec['Modele']==m]['R2'].values[0]) \
            if m in res_avec['Modele'].values else None
        r2_s = float(res_sans[
            res_sans['Modele']==m]['R2'].values[0]) \
            if m in res_sans['Modele'].values else None
        if r2_a and r2_s:
            print(f'{m:25s} | {r2_a:>12.4f} | '
                  f'{r2_s:>12.4f} | {r2_a-r2_s:>+10.4f}')

In [9]:
# ------------------------------------------------------------
# Analyse des résidus par strate de taille de commune
# Objectif : vérifier que le modèle performe aussi bien
# sur les petites communes rurales que les grandes villes
# ------------------------------------------------------------

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Prédictions Extra Trees cible A
y_pred_A = models_A['Extra Trees'].predict(X_te_A)

# Récupérer la population du test
pop_test = test_c[test_c['ofgl_Recettes_totales_log'].notna()][
    'geo_population'].reset_index(drop=True)

# Définir les strates
seuils = [0, 500, 2000, 10000, 50000, float('inf')]
labels_strates = ['< 500 hab', '500-2k', '2k-10k', '10k-50k', '> 50k']

strates = pd.cut(pop_test, bins=seuils, labels=labels_strates)

# Métriques par strate
print("=== PERFORMANCES PAR STRATE (Extra Trees, Cible A) ===")
print(f"{'Strate':15s} | {'N':>6} | {'R2':>7} | {'RMSE':>7} | {'MAE':>7}")
print("-"*50)

strate_results = []
for s in labels_strates:
    mask = strates == s
    if mask.sum() < 10:
        continue
    y_true_s = y_te_A.values[mask]
    y_pred_s = y_pred_A[mask]
    r2   = r2_score(y_true_s, y_pred_s)
    rmse = np.sqrt(mean_squared_error(y_true_s, y_pred_s))
    mae  = mean_absolute_error(y_true_s, y_pred_s)
    n    = mask.sum()
    print(f"{s:15s} | {n:>6,} | {r2:>7.4f} | {rmse:>7.4f} | {mae:>7.4f}")
    strate_results.append({
        'Strate': s, 'N': n, 'R2': r2, 'RMSE': rmse, 'MAE': mae
    })

# Figure — R2 et RMSE par strate
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
BLUE = '#2C5F8A'

df_strates = pd.DataFrame(strate_results)

axes[0].bar(df_strates['Strate'], df_strates['R2'],
            color=BLUE, edgecolor='white')
axes[0].set_xlabel('Strate de population', fontsize=11)
axes[0].set_ylabel('$R^2$', fontsize=11)
axes[0].set_title('$R^2$ par strate de taille',
                  fontsize=11, fontweight='bold')
axes[0].tick_params(axis='x', rotation=20)
for i, row in df_strates.iterrows():
    axes[0].text(i, row['R2'] + 0.005,
                 f"{row['R2']:.3f}", ha='center', fontsize=9)

axes[1].bar(df_strates['Strate'], df_strates['RMSE'],
            color='#E07B39', edgecolor='white')
axes[1].set_xlabel('Strate de population', fontsize=11)
axes[1].set_ylabel('RMSE', fontsize=11)
axes[1].set_title('RMSE par strate de taille',
                  fontsize=11, fontweight='bold')
axes[1].tick_params(axis='x', rotation=20)
for i, row in df_strates.iterrows():
    axes[1].text(i, row['RMSE'] + 0.002,
                 f"{row['RMSE']:.3f}", ha='center', fontsize=9)

plt.suptitle(
    'Figure 8 -- Performances par strate de taille\n'
    '(Extra Trees, Cible A, test 2022-2023)',
    fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig8_performances_par_strate.png',
            dpi=150, bbox_inches='tight')
plt.close()
print("\nfig8 sauvegardee")

# Même analyse pour GBM cible B
print("\n=== PERFORMANCES PAR STRATE (GBM, Cible B) ===")
y_pred_B = models_B['Gradient Boosting'].predict(X_te_B)
pop_test_B = test_c[
    test_c['ofgl_Recettes_totales_pct_hab'].notna()
]['geo_population'].reset_index(drop=True)
strates_B = pd.cut(pop_test_B, bins=seuils, labels=labels_strates)

print(f"{'Strate':15s} | {'N':>6} | {'R2':>7} | {'RMSE':>7} | {'MAE':>7}")
print("-"*50)
for s in labels_strates:
    mask = strates_B == s
    if mask.sum() < 10:
        continue
    y_true_s = y_te_B.values[mask]
    y_pred_s = y_pred_B[mask]
    r2   = r2_score(y_true_s, y_pred_s)
    rmse = np.sqrt(mean_squared_error(y_true_s, y_pred_s))
    mae  = mean_absolute_error(y_true_s, y_pred_s)
    print(f"{s:15s} | {mask.sum():>6,} | {r2:>7.4f} | "
          f"{rmse:>7.4f} | {mae:>7.4f}")

NameError: name 'models_A' is not defined